# AgentCore에 배포된 A2A Agent 및 MCP Server를 Registry에 게시

## 개요

이 Notebook에서는 AWS Agent Registry의 전체 publisher 워크플로를 살펴봅니다. 주문 관리를 위한 **MCP 서버**와 **A2A 에이전트**를 구축하여 **Amazon Bedrock AgentCore Runtime**에 배포하고 정상 작동을 확인합니다. 그런 다음 올바른 descriptor 구조로 **AWS Agent Registry**에 등록하고 승인을 요청한 후 semantic search로 등록된 레코드를 검색합니다.

![아키텍처 흐름](images/agentregistry_flow.png)

## 학습 목표

- 주문 관리를 위한 도구 중심 **MCP 서버**와 에이전트 중심 **A2A 에이전트** 구축 방법
- 두 구성 요소를 **AgentCore Runtime**에 배포하고 작동 여부를 확인하는 방법
- 레코드를 구성할 **Agent Registry** 생성 방법
- 각 프로토콜에 맞는 descriptor 구조(MCP는 `serverSchema` + `toolSchema`, A2A는 `agentCard`)로 **Registry 레코드**를 생성하는 방법
- **승인 워크플로**(DRAFT → PENDING_APPROVAL → APPROVED)의 작동 방식
- Agent Registry에서 **semantic search**를 수행하여 등록된 에이전트와 도구를 검색하는 방법

## 프로토콜 요약

| 구분 | MCP | A2A |
|---|---|---|
| **철학** | 도구/함수 중심 | 에이전트/skill 중심 |
| **Transport** | Streamable HTTP | HTTP 기반 JSON-RPC 2.0 |
| **Port** | 8000 | 9000 |
| **Endpoint mount** | `/mcp` | `/` (root) |
| **검색** | `POST /mcp` → `tools/list` | `GET /.well-known/agent-card.json` |
| **호출** | `POST /mcp` → `tools/call` | `POST /` → `message/send` |
| **Registry descriptor** | `mcp.serverSchema` + `mcp.toolSchema` | `a2a.agentCard` |
| **State model** | Stateless 함수 호출 | 메시지 기록이 있는 stateful 작업 |
| **AgentCore 인증** | SigV4 | SigV4 또는 OAuth 2.0 |

---
## 설정

### 사전 요구 사항

- IAM 자격 증명이 구성된 AWS 계정
- Python 3.10+
- `boto3 >= 1.42.87`
- 다음 권한이 있는 IAM 사용자 또는 role(`ACCOUNT_ID`와 `REGION`은 필요에 따라 변경)
- IAM 사용자 또는 role에는 AgentCore Runtime에 배포할 적절한 권한도 있어야 합니다. 자세한 내용은 [AgentCore Runtime 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html)를 참조하세요.

<details>
<summary>필수 IAM policy(클릭하여 펼치기)</summary>

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowCreateRegistry",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:CreateRegistry"],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:*"]
        },
        {
            "Sid": "AllowGetUpdateDeleteRegistry",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistry",
                "bedrock-agentcore:DeleteRegistry"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowCreateAndListRecords",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateRegistryRecord",
                "bedrock-agentcore:SearchRegistryRecords"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowRecordOperations",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistryRecord",
                "bedrock-agentcore:DeleteRegistryRecord",
                "bedrock-agentcore:SubmitRegistryRecordForApproval"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:registry/*/record/*"]
        }
    ]
}
```

</details>

**참고:** 이 Notebook은 에이전트 배포에 `bedrock-agentcore-starter-toolkit`, `strands-agents`, `mcp`를 사용합니다. 이 패키지는 `requirements.txt`를 통해 자동으로 설치됩니다.

### 종속성 설치

In [ ]:
!pip install -r requirements.txt

### AWS 세션 및 클라이언트 초기화

AgentCore Runtime에 SigV4 서명 요청을 보내기 위한 boto3 세션과 Agent Registry 관리 및 검색 작업을 위한 boto3 클라이언트가 필요합니다.

In [ ]:
from boto3.session import Session
import json
import time
import uuid
import os

# 구성
boto_session = Session()
AWS_REGION = boto_session.region_name

# AWS_PROFILE = "aws-profile"  # 사용자의 profile로 변경하세요. SageMaker에서 실행하는 경우 이 줄을 주석 처리하세요.

# Amazon SageMaker Notebook을 사용하지 않는 경우 AWS 자격 증명 설정
# os.environ["AWS_PROFILE"] = AWS_PROFILE

# boto3 세션 생성
# boto_session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)  # SageMaker에서 실행하지 않는 경우 사용하세요.

# Registry 관리용 클라이언트
registry_client = boto_session.client("bedrock-agentcore-control", region_name=AWS_REGION)

# 검색용 클라이언트
search_client = boto_session.client("bedrock-agentcore", region_name=AWS_REGION)

# 에이전트 디렉터리 생성
os.makedirs("agents/mcp", exist_ok=True)
os.makedirs("agents/a2a", exist_ok=True)

print(f"Session ready | Region: {AWS_REGION}")

### Helper 함수

보기 좋은 출력, AgentCore Runtime에 대한 SigV4 서명 HTTP 요청 및 Registry 상태 폴링에 사용할 utility입니다.

In [ ]:
import requests
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest


# Terminal 출력용 ANSI 색상
class C:
    GREEN = "\033[92m"
    RED = "\033[91m"
    YELLOW = "\033[93m"
    CYAN = "\033[96m"
    BOLD = "\033[1m"
    DIM = "\033[2m"
    RESET = "\033[0m"


def pretty_print_response(response):
    """API 응답에서 ResponseMetadata를 제외하고 보기 좋게 출력합니다."""
    data = {k: v for k, v in response.items() if k != "ResponseMetadata"}
    print(json.dumps(data, indent=2, default=str))


def signed_mcp_post(url, payload):
    """SigV4로 서명한 MCP JSON-RPC POST를 보내고 JSON 또는 SSE 응답을 파싱합니다."""
    credentials = boto_session.get_credentials().get_frozen_credentials()
    data = json.dumps(payload)
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    req = AWSRequest(method="POST", url=url, data=data, headers=headers)
    SigV4Auth(credentials, "bedrock-agentcore", AWS_REGION).add_auth(req)
    resp = requests.post(url, headers=dict(req.headers), data=data, timeout=30)
    text = resp.text.strip()
    if text.startswith("{") or text.startswith("["):
        return json.loads(text)
    for line in text.splitlines():
        if line.startswith("data:"):
            return json.loads(line[len("data:") :].strip())
    raise ValueError(f"Could not parse MCP response (status {resp.status_code}): {text[:300]}")


def signed_get(url):
    """A2A 에이전트 카드 조회에 사용할 SigV4 서명 GET 요청을 보냅니다."""
    credentials = boto_session.get_credentials().get_frozen_credentials()
    headers = {"Accept": "*/*"}
    req = AWSRequest(method="GET", url=url, headers=headers)
    SigV4Auth(credentials, "bedrock-agentcore", AWS_REGION).add_auth(req)
    return requests.get(url, headers=dict(req.headers), timeout=30)


def signed_a2a_post(url, payload):
    """SigV4로 서명한 A2A JSON-RPC POST를 보내고 응답을 파싱합니다."""
    credentials = boto_session.get_credentials().get_frozen_credentials()
    data = json.dumps(payload)
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    req = AWSRequest(method="POST", url=url, data=data, headers=headers)
    SigV4Auth(credentials, "bedrock-agentcore", AWS_REGION).add_auth(req)
    resp = requests.post(url, headers=dict(req.headers), data=data, timeout=30)
    text = resp.text.strip()
    if text.startswith("{") or text.startswith("["):
        return json.loads(text)
    for line in text.splitlines():
        if line.startswith("data:"):
            return json.loads(line[len("data:") :].strip())
    raise ValueError(f"Could not parse A2A response (status {resp.status_code}): {text[:300]}")


def wait_for_registry(registry_id, interval=5):
    """레지스트리가 READY 상태가 될 때까지 폴링합니다."""
    while True:
        resp = registry_client.get_registry(registryId=registry_id)
        status = resp["status"]
        if status == "READY":
            print(f"  {C.GREEN}✅ Registry Status: {status}{C.RESET}")
            return resp
        if status.endswith("_FAILED"):
            print(f"  {C.RED}❌ Registry Status: {status}{C.RESET}")
            raise Exception(f"Registry failed: {status} - {resp.get('statusReason')}")
        print(f"  {C.YELLOW}⏳ Registry Status: {status}{C.RESET}")
        time.sleep(interval)

---
## 1. MCP Server를 AgentCore Runtime에 배포

먼저 주문 관리 도구가 있는 MCP 서버를 AgentCore Runtime에 배포합니다. 배포 후 `tools/list`와 `tools/call`을 호출하여 작동 여부를 확인합니다. 이 응답은 나중에 Registry 레코드를 생성할 때도 사용합니다.

```
Client → POST /mcp (AgentCore Runtime, port 8000)
  ├─ tools/list                              → [{name, description, inputSchema}, ...]
  └─ tools/call  {name, arguments}           → {content: [{type: "text", text: "..."}]}
```

### 1.1 MCP Server 작성

`stateless_http=True`로 `FastMCP`를 사용합니다. 이 서버는 주문 생성, 조회, 업데이트, 취소 및 목록 조회를 위한 주문 관리 도구를 제공합니다.

In [ ]:
%%writefile agents/mcp/mcp_order_server.py
"""FastMCP를 통해 주문 CRUD 도구를 제공하는 MCP 주문 관리 서버입니다.

프로토콜: MCP (Model Context Protocol)
포트:     8000(streamable-http 기본값)
마운트:   /mcp
검색:     POST /mcp -> tools/list
호출:     POST /mcp -> tools/call
"""
import uuid
from datetime import datetime
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(
    name="order-management-tools",
    instructions="A collection of order management tools for creating, updating, and managing orders.",
    host="0.0.0.0",
    stateless_http=True,
)


@mcp.tool()
def create_order(customer_name: str, product: str, quantity: int) -> str:
    """Create a new order for a customer."""
    order_id = f"ORD-{uuid.uuid4().hex[:8].upper()}"
    return (
        f"Order created successfully. "
        f"Order ID: {order_id}, Customer: {customer_name}, "
        f"Product: {product}, Quantity: {quantity}, "
        f"Status: PENDING, Created: {datetime.now().isoformat()}"
    )


@mcp.tool()
def get_order(order_id: str) -> str:
    """Retrieve details of an existing order by its ID."""
    return (
        f"Order ID: {order_id}, Customer: Jane Smith, "
        f"Product: Wireless Headphones, Quantity: 2, "
        f"Status: SHIPPED, Total: $149.98, "
        f"Created: 2025-01-15T10:30:00, Shipped: 2025-01-16T14:00:00"
    )


@mcp.tool()
def update_order(order_id: str, quantity: int = None, product: str = None) -> str:
    """Update an existing order's quantity or product."""
    updates = []
    if quantity is not None:
        updates.append(f"Quantity: {quantity}")
    if product is not None:
        updates.append(f"Product: {product}")
    return (
        f"Order {order_id} updated successfully. "
        f"Changes: {', '.join(updates) if updates else 'None'}, "
        f"Updated: {datetime.now().isoformat()}"
    )


@mcp.tool()
def cancel_order(order_id: str, reason: str) -> str:
    """Cancel an existing order with a reason."""
    return (
        f"Order {order_id} cancelled successfully. "
        f"Reason: {reason}, Status: CANCELLED, "
        f"Cancelled: {datetime.now().isoformat()}"
    )


@mcp.tool()
def list_orders(status: str = "ALL") -> str:
    """List orders, optionally filtered by status (PENDING, SHIPPED, DELIVERED, CANCELLED, ALL)."""
    return (
        f"Orders (filter: {status}):\n"
        f"  1. ORD-A1B2C3D4 | Jane Smith    | Wireless Headphones (2) | SHIPPED\n"
        f"  2. ORD-E5F6G7H8 | John Doe      | USB-C Cable (5)        | PENDING\n"
        f"  3. ORD-I9J0K1L2 | Alice Johnson | Laptop Stand (1)       | DELIVERED"
    )


if __name__ == "__main__":
    mcp.run(transport="streamable-http")


### 1.2 AgentCore Runtime에 배포

`bedrock-agentcore-starter-toolkit`은 서버 코드를 container로 패키징하여 ECR에 push하고 AgentCore Runtime endpoint를 생성합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

mcp_runtime = Runtime()

print("Configuring MCP AgentCore Runtime...")
mcp_runtime.configure(
    agent_name="mcp_order_server",
    protocol="MCP",
    entrypoint="agents/mcp/mcp_order_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="agents/mcp/requirements.txt",
    region=AWS_REGION,
)
print(f"  {C.GREEN}✅ Configuration completed{C.RESET}")

In [ ]:
print("Launching MCP agent to AgentCore Runtime...")
print("This may take several minutes...")
mcp_launch_result = mcp_runtime.launch()

mcp_agent_arn = mcp_launch_result.agent_arn
mcp_agent_id = mcp_launch_result.agent_id

mcp_encoded_arn = mcp_agent_arn.replace(":", "%3A").replace("/", "%2F")
MCP_ENDPOINT_URL = (
    f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{mcp_encoded_arn}/invocations?qualifier=DEFAULT"
)

print(f"  {C.GREEN}✅ Launch completed{C.RESET}")
print(f"  {C.BOLD}ARN:{C.RESET}      {C.CYAN}{mcp_agent_arn}{C.RESET}")
print(f"  {C.BOLD}Endpoint:{C.RESET}  {C.CYAN}{MCP_ENDPOINT_URL}{C.RESET}")

### 1.3 배포 확인 - `tools/list`

`tools/list`를 호출하여 MCP 서버가 실행 중인지 확인하고 제공되는 도구를 검색합니다. 이 응답은 Agent Registry에 등록할 때 `toolSchema`로 사용됩니다.

In [ ]:
# 배포된 MCP 서버의 도구 검색
mcp_tools_response = signed_mcp_post(MCP_ENDPOINT_URL, {"jsonrpc": "2.0", "id": 1, "method": "tools/list"})

print(f"{C.BOLD}MCP tools/list response:{C.RESET}")
pretty_print_response(mcp_tools_response)

### 1.4 호출 테스트 - `tools/call`

서버를 등록하기 전에 도구를 호출하여 올바르게 작동하는지 확인합니다.

In [ ]:
# 'create_order' 도구 호출
mcp_invoke_response = signed_mcp_post(
    MCP_ENDPOINT_URL,
    {
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": {
            "name": "create_order",
            "arguments": {
                "customer_name": "Jane Smith",
                "product": "Wireless Headphones",
                "quantity": 2,
            },
        },
    },
)

print(f"{C.BOLD}MCP tools/call response:{C.RESET}")
pretty_print_response(mcp_invoke_response)

---
## 2. A2A Agent를 AgentCore Runtime에 배포

다음으로 동일한 주문 관리 기능을 가진 A2A 에이전트를 배포합니다. 배포 후 agent card를 가져와 호출을 테스트합니다. Agent card는 Registry 레코드 descriptor로 사용됩니다.

```
Client → AgentCore Runtime (port 9000)
  ├─ GET  /.well-known/agent-card.json       → {name, skills, capabilities}
  ├─ POST / (message/send)  {message}       → {task result with message parts}
  └─ POST / (JSON-RPC 2.0)  {jsonrpc, method, params}  → {jsonrpc, result}
```

### 2.1 A2A Server 작성

[AgentCore 공식 A2A 배포 가이드](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-a2a.html)에 따라 `FastAPI`, `uvicorn`과 함께 Strands `A2AServer`를 사용합니다. 에이전트는 주문 관리 기능을 자연어 skill로 제공합니다.

In [ ]:
%%writefile agents/a2a/a2a_order_agent.py
"""AgentCore Runtime에 배포하는 A2A 주문 관리 에이전트입니다.

프로토콜: A2A (Agent-to-Agent)
포트:     9000(AgentCore의 A2A 기본값)
마운트:   /(루트)
검색:     GET /.well-known/agent-card.json
호출:     POST / -> message/send
"""
import os
import uuid as _uuid
from datetime import datetime
from strands import Agent, tool
from strands.multiagent.a2a import A2AServer
from a2a.types import AgentSkill
from fastapi import FastAPI
import uvicorn

runtime_url = os.environ.get("AGENTCORE_RUNTIME_URL", "http://127.0.0.1:9000/")
host, port = "0.0.0.0", 9000


@tool
def create_order(customer_name: str, product: str, quantity: int) -> str:
    """Create a new order for a customer."""
    order_id = f"ORD-{_uuid.uuid4().hex[:8].upper()}"
    return (
        f"Order created successfully. "
        f"Order ID: {order_id}, Customer: {customer_name}, "
        f"Product: {product}, Quantity: {quantity}, "
        f"Status: PENDING, Created: {datetime.now().isoformat()}"
    )


@tool
def get_order(order_id: str) -> str:
    """Retrieve details of an existing order by its ID."""
    return (
        f"Order ID: {order_id}, Customer: Jane Smith, "
        f"Product: Wireless Headphones, Quantity: 2, "
        f"Status: SHIPPED, Total: $149.98, "
        f"Created: 2025-01-15T10:30:00, Shipped: 2025-01-16T14:00:00"
    )


@tool
def update_order(order_id: str, quantity: int = None, product: str = None) -> str:
    """Update an existing order's quantity or product."""
    updates = []
    if quantity is not None:
        updates.append(f"Quantity: {quantity}")
    if product is not None:
        updates.append(f"Product: {product}")
    return (
        f"Order {order_id} updated successfully. "
        f"Changes: {', '.join(updates) if updates else 'None'}, "
        f"Updated: {datetime.now().isoformat()}"
    )


@tool
def cancel_order(order_id: str, reason: str) -> str:
    """Cancel an existing order with a reason."""
    return (
        f"Order {order_id} cancelled successfully. "
        f"Reason: {reason}, Status: CANCELLED, "
        f"Cancelled: {datetime.now().isoformat()}"
    )


@tool
def list_orders(status: str = "ALL") -> str:
    """List orders, optionally filtered by status."""
    return (
        f"Orders (filter: {status}):\n"
        f"  1. ORD-A1B2C3D4 | Jane Smith    | Wireless Headphones (2) | SHIPPED\n"
        f"  2. ORD-E5F6G7H8 | John Doe      | USB-C Cable (5)        | PENDING\n"
        f"  3. ORD-I9J0K1L2 | Alice Johnson | Laptop Stand (1)       | DELIVERED"
    )


agent = Agent(
    system_prompt=(
        "You are an order management assistant. Use the available tools to "
        "create, retrieve, update, cancel, and list orders. "
        "Be concise and confirm actions clearly."
    ),
    tools=[create_order, get_order, update_order, cancel_order, list_orders],
    name="order-management-agent",
    description="An order management agent that handles order creation, updates, cancellations, and lookups",
)

a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True,
    skills=[
        AgentSkill(
            id="order-management",
            name="Order Management",
            description="Create, retrieve, update, and cancel customer orders",
            examples=["Create an order for 2 headphones for Jane Smith", "Cancel order ORD-A1B2C3D4"],
            tags=[],
        ),
        AgentSkill(
            id="order-tracking",
            name="Order Tracking",
            description="Look up order status and list orders by status",
            examples=["What is the status of order ORD-A1B2C3D4?", "Show me all pending orders"],
            tags=[],
        ),
    ],
)

app = FastAPI()


@app.get("/ping")
def ping():
    return {"status": "healthy"}


app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)


### 2.2 AgentCore Runtime에 배포

MCP와 동일한 배포 흐름을 사용하되 `protocol="A2A"`로 설정합니다. AgentCore Runtime은 A2A traffic을 root `/`의 9000번 port로 route합니다.

In [ ]:
# 충돌을 방지하기 위해 MCP 에이전트의 YAML 구성 제거
import os

if os.path.exists(".bedrock_agentcore.yaml"):
    os.remove(".bedrock_agentcore.yaml")
    print(f"  {C.YELLOW}⏳ Cleared previous agent config{C.RESET}")
if os.path.exists("Dockerfile"):
    os.remove("Dockerfile")
    print(f"  {C.YELLOW}⏳ Cleared previous docker file{C.RESET}")

a2a_runtime = Runtime()

print("Configuring A2A AgentCore Runtime...")
a2a_runtime.configure(
    agent_name="a2a_order_agent",
    protocol="A2A",
    entrypoint="agents/a2a/a2a_order_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="agents/a2a/requirements.txt",
    region=AWS_REGION,
)
print(f"  {C.GREEN}✅ Configuration completed{C.RESET}")

In [ ]:
print("Launching A2A agent to AgentCore Runtime...")
print("This may take several minutes...")
a2a_launch_result = a2a_runtime.launch()

a2a_agent_arn = a2a_launch_result.agent_arn
a2a_agent_id = a2a_launch_result.agent_id

a2a_encoded_arn = a2a_agent_arn.replace(":", "%3A").replace("/", "%2F")
A2A_ENDPOINT_URL = (
    f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{a2a_encoded_arn}/invocations?qualifier=DEFAULT"
)

print(f"  {C.GREEN}✅ Launch completed{C.RESET}")
print(f"  {C.BOLD}ARN:{C.RESET}      {C.CYAN}{a2a_agent_arn}{C.RESET}")
print(f"  {C.BOLD}Endpoint:{C.RESET}  {C.CYAN}{A2A_ENDPOINT_URL}{C.RESET}")

### 2.3 배포 확인 - Agent Card

배포된 A2A 서버에서 agent card를 가져옵니다. 이 JSON 문서는 에이전트의 기능과 skill을 설명합니다. Agent Registry에 등록할 때 `agentCard` descriptor로 사용됩니다.

Agent card URL은 다음 pattern을 따릅니다: `/runtimes/{encoded-ARN}/invocations/.well-known/agent-card.json`

In [ ]:
# 배포된 A2A 서버에서 실제 agent card 가져오기
agent_card_url = f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{a2a_encoded_arn}/invocations/.well-known/agent-card.json"

agent_card_response = signed_get(agent_card_url)
a2a_agent_card = agent_card_response.json()

print(f"{C.BOLD}A2A Agent Card (from GET /.well-known/agent-card.json):{C.RESET}")
pretty_print_response(a2a_agent_card)

### 2.4 호출 테스트 - `message/send`

에이전트를 등록하기 전에 자연어 메시지를 전송하여 올바르게 작동하는지 확인합니다.

In [ ]:
# 자연어 메시지로 A2A 에이전트 호출
a2a_invoke_payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "message/send",
    "params": {
        "message": {
            "role": "user",
            "messageId": str(uuid.uuid4()),
            "parts": [
                {
                    "kind": "text",
                    "text": "Create an order for 2 Wireless Headphones for Jane Smith",
                }
            ],
        }
    },
}

a2a_invoke_response = signed_a2a_post(A2A_ENDPOINT_URL, a2a_invoke_payload)

print(f"{C.BOLD}A2A message/send response:{C.RESET}")
pretty_print_response(a2a_invoke_response)

---
## 3. Agent Registry에 등록

두 에이전트의 배포와 확인을 마쳤으므로 Agent Registry에 등록합니다. 각 프로토콜은 서로 다른 descriptor 구조를 사용합니다.

- **MCP 레코드**는 `serverSchema`(server metadata) + `toolSchema`(`tools/list`의 함수 정의)를 사용합니다.
- **A2A 레코드**는 `agentCard`(`/.well-known/agent-card.json`의 agent card 문서)를 사용합니다.

레코드가 검색 가능해지기 전에 승인 워크플로를 거치도록 `autoApproval`을 `False`로 설정합니다.

### 3.1 Registry 생성

In [ ]:
create_registry_respone = registry_client.create_registry(
    name="agentcore-tools-registry",
    description="Registry to store A2A Agents and MCP Servers deployed on AgentCore",
    approvalConfiguration={"autoApproval": False},
)

REGISTRY_ARN = create_registry_respone["registryArn"]
REGISTRY_ID = REGISTRY_ARN.split("/")[-1]

wait_for_registry(REGISTRY_ID)

print(f"  {C.GREEN}✅ Registry created!{C.RESET}")
print(f"  {C.BOLD}ARN:{C.RESET}  {C.CYAN}{REGISTRY_ARN}{C.RESET}")
print(f"  {C.BOLD}ID:{C.RESET}   {C.CYAN}{REGISTRY_ID}{C.RESET}")

### 3.2 MCP Registry 레코드 생성

MCP descriptor에는 다음 항목이 포함됩니다.
- `serverSchema`: 서버에 대한 OpenAPI 스타일 metadata(name, description, packages, transports)
- `toolSchema`: Input/output schema가 포함된 개별 함수 정의

In [ ]:
# 실제 tools/list 응답(셀 1.3)에서 tool schema 구성
mcp_tools = mcp_tools_response.get("result", {}).get("tools", [])
mcp_tool_schema = json.dumps({"tools": mcp_tools})

# Package metadata가 포함된 server schema
mcp_server_schema = json.dumps(
    {
        "name": "io.example/order-management-tools",
        "description": "MCP server exposing order management tools",
        "version": "1.0.0",
        "title": "Order Management MCP Server",
        "packages": [
            {
                "registryType": "pypi",
                "identifier": "order-mcp-server",
                "version": "1.0.0",
                "runtimeHint": "python",
                "transport": {"type": "stdio"},
            }
        ],
    }
)

mcp_record_respone = registry_client.create_registry_record(
    registryId=REGISTRY_ID,
    name="order_mcp_server",
    description="MCP server with order management tools",
    descriptorType="MCP",
    recordVersion="1.0",
    descriptors={
        "mcp": {
            "server": {
                "schemaVersion": "2025-12-11",
                "inlineContent": mcp_server_schema,
            },
            "tools": {
                "protocolVersion": "2025-11-25",
                "inlineContent": mcp_tool_schema,
            },
        }
    },
)

MCP_RECORD_ID = mcp_record_respone["recordArn"].split("/")[-1]
print(f"  {C.GREEN}✅ MCP record created: {C.CYAN}{MCP_RECORD_ID}{C.RESET}")

### 3.3 A2A Registry 레코드 생성

A2A descriptor에는 다음 항목이 포함됩니다.
- `agentCard`: 에이전트 전체를 설명하는 A2A agent card 문서(skill, 기능, endpoint URL)
- 함수별 schema 없음: Agent card의 skill은 자연어를 사용합니다.

In [ ]:
# 배포된 A2A 서버에서 가져온 실제 agent card(셀 2.3) 사용
a2a_agent_card_schema = json.dumps(a2a_agent_card)

a2a_record_respone = registry_client.create_registry_record(
    registryId=REGISTRY_ID,
    name="order_a2a_agent",
    description="A2A agent for managing customer orders conversationally",
    descriptorType="A2A",
    recordVersion="1.0",
    descriptors={
        "a2a": {
            "agentCard": {
                "schemaVersion": a2a_agent_card.get("protocolVersion", "0.3.0"),
                "inlineContent": a2a_agent_card_schema,
            }
        }
    },
)

A2A_RECORD_ID = a2a_record_respone["recordArn"].split("/")[-1]
print(f"  {C.GREEN}✅ A2A record created: {C.CYAN}{A2A_RECORD_ID}{C.RESET}")

---
## 4. 승인 워크플로

레코드는 검색 결과에 표시되기 전에 승인을 받아야 합니다. 승인 워크플로는 표준 Registry pattern을 따릅니다.

```
DRAFT → PENDING_APPROVAL → APPROVED (이제 검색 가능)
                         → REJECTED
                         → DEPRECATED (승인 후 사용 중단 시)
```

프로덕션 환경에서는 publisher가 레코드를 제출하고 별도의 admin이 검토하고 승인합니다. 여기서는 두 단계를 모두 수행합니다.

### 4.1 레코드가 DRAFT 상태로 시작하는지 확인

In [ ]:
records_response = registry_client.list_registry_records(registryId=REGISTRY_ID)

print(f"{C.BOLD}=== Registry Records ==={C.RESET}")
print(f"Found {len(records_response['registryRecords'])} record(s):\n")
for rec in records_response["registryRecords"]:
    status = rec["status"]
    sc = C.GREEN if status == "APPROVED" else C.YELLOW if status in ("DRAFT", "PENDING_APPROVAL") else C.RED
    print(f"  {sc}[{status}]{C.RESET} {rec['name']} | {C.DIM}{rec['recordId']}{C.RESET}")

### 4.2 승인 요청 제출 및 승인

두 레코드의 승인 요청을 제출한 후 승인합니다. 승인된 레코드는 Agent Registry에서 검색할 수 있습니다.

In [ ]:
for record_id, record_name in [(MCP_RECORD_ID, "MCP"), (A2A_RECORD_ID, "A2A")]:
    # 1단계: Publisher가 승인 요청 제출
    registry_client.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=record_id)
    print(f"  {C.YELLOW}⏳ {record_name} record → PENDING_APPROVAL{C.RESET}")

    # 2단계: Admin이 승인
    registry_client.update_registry_record_status(
        registryId=REGISTRY_ID,
        recordId=record_id,
        statusReason="Approved by admin",
        status="APPROVED",
    )
    print(f"  {C.GREEN}✅ {record_name} record → APPROVED{C.RESET}")

### 4.3 APPROVED 상태 확인

In [ ]:
for record_id, record_name in [(MCP_RECORD_ID, "MCP"), (A2A_RECORD_ID, "A2A")]:
    rec = registry_client.get_registry_record(registryId=REGISTRY_ID, recordId=record_id)
    status = rec["status"]
    sc = C.GREEN if status == "APPROVED" else C.RED
    print(f"  {sc}{record_name} record status: {status}{C.RESET}")
    assert status == "APPROVED", f"Expected APPROVED, got {status}"

---
## 5. Semantic Search

두 레코드가 승인되어 index에 포함되었으므로 Agent Registry에서 검색할 수 있습니다. 자연어 query를 사용하여 등록된 MCP 서버와 A2A 에이전트를 검색합니다.

### 5.1 주문 관리 도구 검색

자연어 query를 사용하여 Registry를 검색합니다. 레코드 name, description 및 descriptor content를 대상으로 semantic matching을 수행합니다.

In [ ]:
# 승인 후 검색 index가 업데이트될 때까지 대기
print(f"  {C.YELLOW}⏳ Waiting for search index to update...{C.RESET}")
time.sleep(100)

In [ ]:
# 주문 관리 도구 검색
search_response = search_client.search_registry_records(
    registryIds=[REGISTRY_ARN], searchQuery="order management", maxResults=5
)
search_response.pop("ResponseMetadata", None)

records = search_response.get("registryRecords", [])
print(f"{C.BOLD}Search results for 'order management':{C.RESET}")
print(f"Found {len(records)} record(s):\n")

for rec in records:
    descriptor_type = rec.get("descriptorType", "N/A")
    name = rec.get("name", "N/A")
    desc = rec.get("description", "")
    status = rec.get("status", "N/A")
    sc = C.GREEN if status == "APPROVED" else C.YELLOW
    print(f"  {sc}[{status}]{C.RESET} {C.CYAN}{name}{C.RESET} ({descriptor_type})")
    print(f"    {desc}")

    descriptors = rec.get("descriptors", {})

    # MCP 도구 표시
    mcp_desc = descriptors.get("mcp", {})
    tool_schema = mcp_desc.get("tools", {}).get("inlineContent", "")
    if tool_schema:
        try:
            tools = json.loads(tool_schema).get("tools", [])
            print(f"    {C.BOLD}Tools:{C.RESET}")
            for t in tools:
                print(f"      • {t['name']}: {t.get('description', '')}")
        except (json.JSONDecodeError, TypeError):
            pass

    # A2A skill 표시
    a2a_desc = descriptors.get("a2a", {})
    agent_card_content = a2a_desc.get("agentCard", {}).get("inlineContent", "")
    if agent_card_content:
        try:
            card = json.loads(agent_card_content)
            skills = card.get("skills", [])
            if skills:
                print(f"    {C.BOLD}Skills:{C.RESET}")
                for s in skills:
                    print(f"      • {s.get('name', s.get('id', '?'))}: {s.get('description', '')}")
        except (json.JSONDecodeError, TypeError):
            pass

### 5.2 다양한 Query로 검색

다양한 자연어 query를 사용하여 semantic search가 등록된 레코드와 일치시키는 방식을 확인합니다.

In [ ]:
# 더 구체적인 query로 검색
for query in [
    "cancel an order",
    "track shipment status",
    "create new order for customer",
]:
    response = search_client.search_registry_records(registryIds=[REGISTRY_ARN], searchQuery=query, maxResults=3)
    records = response.get("registryRecords", [])
    print(f"{C.BOLD}'{query}'{C.RESET} → {len(records)} result(s)")
    for rec in records:
        print(f"  • {rec.get('name', 'N/A')} ({rec.get('descriptorType', 'N/A')})")
    print()

---
## 6. 정리(선택 사항)

AgentCore Runtime 에이전트, Registry 레코드 및 Registry를 삭제하여 리소스를 정리합니다.

In [ ]:
# AgentCore Runtime 에이전트 삭제
agentcore_client = boto_session.client("bedrock-agentcore-control", region_name=AWS_REGION)

for agent_id, agent_name in [(mcp_agent_id, "MCP"), (a2a_agent_id, "A2A")]:
    try:
        agentcore_client.delete_agent_runtime(agentRuntimeId=agent_id)
        print(f"  {C.GREEN}✅ Deleted {agent_name} runtime: {C.DIM}{agent_id}{C.RESET}")
    except Exception as e:
        print(f"  {C.RED}❌ Failed to delete {agent_name} runtime: {e}{C.RESET}")

# Registry의 모든 레코드 삭제
records = registry_client.list_registry_records(registryId=REGISTRY_ID)
for rec in records.get("registryRecords", []):
    record_id = rec["recordId"]
    registry_client.delete_registry_record(registryId=REGISTRY_ID, recordId=record_id)
    print(f"  {C.GREEN}✅ Deleted record: {C.DIM}{record_id}{C.RESET}")

# Registry 삭제
registry_client.delete_registry(registryId=REGISTRY_ID)
print(f"  {C.GREEN}✅ Deleted registry: {C.DIM}{REGISTRY_ID}{C.RESET}")

print(f"\n  {C.GREEN}✅ Cleanup complete!{C.RESET}")